# STEP 4 — SUBTITLE GENERATION (Text to Subtitle)


## Useful Information

**Subtitle Generation**

Narration Script  
↓  
Subtitle Chunks  
↓  
Forced Alignment (word timestamps)  
↓  
Subtitle Segments  
↓  
SRT File

The narration script is split into short chunks that fit on screen (at most `MAX_CHARS_PER_SUBTITLE` = 42 characters). Each chunk is then given a start and end time from the narration audio and written to an `.srt` file.

**Forced Alignment**

Narration Audio  
↓  
Whisper (`whisper-1`)  
↓  
Word-level Timestamps  
↓  
Matched with Subtitle Chunks

Forced alignment finds when each word is spoken in the audio. Whisper transcribes the narration audio with `timestamp_granularities=["word"]`, and each subtitle chunk takes its start time from its first word and its end time from its last word.

**SRT (SubRip Subtitle) Format**

Index  
↓  
Start Time --> End Time  
↓  
Subtitle Text

**SRT (SubRip Subtitle)** is a common subtitle file format that stores the subtitle number, start and end timestamps, and subtitle text. Each block has an index, a time range in `HH:MM:SS,mmm` format, and the text to show, for example:

```
1
00:00:00,000 --> 00:00:01,550
AI is changing the world
```

## 1. Setup

### 1.1 Import Libraries

In [1]:
import os
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

### 1.2 Environment

In [2]:
load_dotenv()

API_KEY = os.getenv("AI_MODEL_API_KEY")

if not API_KEY:
    raise ValueError("AI_MODEL_API_KEY is not set.")

### 1.3 Paths

In [3]:
OUTPUT_DIR = Path("../outputs")
SUBTITLE_DIR = OUTPUT_DIR / "subtitles"

SUBTITLE_DIR.mkdir(parents = True, exist_ok = True)

### 1.4 Define Subtitle Settings

In [4]:
MAX_CHARS_PER_SUBTITLE = 42

ALIGNMENT_PROVIDER = "openai"
ALIGNMENT_MODEL = "whisper-1"

## 2. Subtitle Models

### 2.1 Define VideoScript

In [5]:
class VideoScript(BaseModel):
    title: str
    narration: str

### 2.2 Define SpeechResult

In [6]:
class SpeechResult(BaseModel):
    audio_path: str
    duration_seconds: float
    provider: str
    voice: str

### 2.3 Define SubtitleSegment

### Subtitle Segmentation

- Subtitle segmentation splits the full narration/script into smaller pieces.

- Each piece appears on screen as an individual subtitle.

In [7]:
class SubtitleSegment(BaseModel):
    index: int      # Based on the number of the subtitle segment
    start_time: float
    end_time: float
    text: str

### 2.4 Define SubtitleResult

In [8]:
class SubtitleResult(BaseModel):
    subtitle_path: str
    segments: list[SubtitleSegment]
    duration_seconds: float

## 3. Subtitle Segmentation

### 3.1 Split Narration into Subtitle Chunks

In [9]:
def split_into_subtitle_chunks(text: str, max_chars: int = MAX_CHARS_PER_SUBTITLE) -> list[str]:
    """Split narration text (full script) into readable subtitle chunks."""

    words = text.split()

    chunks = []             # to store all chunks
    current_chunk = []      # keep track of the current_chunk

    for word in words:
        candidate = " ".join(current_chunk + [word])    # " ".join() combines multiple strings into one string, putting a space " " between them.

        # Add the whole word at the character level
        if len(candidate) <= max_chars:                  # if the current_chunk has fewer than 42 characters
            current_chunk.append(word)
        else:
            chunks.append(" ".join(current_chunk))

            current_chunk = [word]
            
    # after the loop finishes, there may still be some words left in current_chunk that were never added to chunks.
    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

### 3.2 Test Subtitle Segmentation

In [10]:
test_narration = ("Artificial intelligence is changing how people work and create. "
                  "AI can help people solve problems and generate new ideas."
                  )

In [11]:
test_chunks = split_into_subtitle_chunks(test_narration, max_chars = 42)

for i, chunk in enumerate(test_chunks, start = 1):
    print(f"{i}: {chunk}")

1: Artificial intelligence is changing how
2: people work and create. AI can help people
3: solve problems and generate new ideas.


## 4. Forced Alignment

### 4.1 Create Alignment Client

In [12]:
def create_alignment_client(provider: str,api_key: str | None):
    """Create a client for the selected alignment provider."""

    if provider == "openai":
        return OpenAI(api_key = api_key)

    elif provider == "deepgram":
        from deepgram import DeepgramClient
        return DeepgramClient(api_key)

    elif provider == "assemblyai":
        import assemblyai as aai
        aai.settings.api_key = api_key
        return aai

    # This function can be extended to support additional alignment providers in the future.

    raise ValueError(f"Unsupported alignment provider: {provider}")

### 4.2 Transcribe Audio with Word Timestamps

In [13]:
# An alignment model figures out when each word in a known text is spoken in an audio recording.

def transcribe_audio(audio_path: str) -> list[dict]:
    """Transcribe narration audio and return word timestamps."""

    with open(audio_path, "rb") as audio_file:  # e.g. mp3 file
        transcript = alignment_client.audio.transcriptions.create(model = ALIGNMENT_MODEL,
                                                                    file = audio_file,
                                                                    response_format = "verbose_json",   # response_format="verbose_json" asks for a detailed structured response 
                                                                    timestamp_granularities = ["word"]    # Give me timestamps at the individual word level.
                                                                    )

    return [word.model_dump() for word in transcript.words]

### How `transcribe_audio()` returns word timestamps

- `transcript.words` is the list of words Whisper heard in the audio.

- Each item is a word object, conceptually like:

```python
TranscriptionWord(word="Artificial", start=0.0, end=0.52)
```

- `word` is the spoken word; `start` and `end` are when it is spoken, in seconds from the start of the audio.

- `word.model_dump()` converts the object into a plain dictionary:

```python
{"word": "Artificial", "start": 0.0, "end": 0.52}
```

- `transcribe_audio()` does this for every word, so it returns a `list[dict]`.

### 4.3 Match Subtitle Chunks with Word Timestamps

In [14]:
def align_chunks_with_words(chunks: list[str], word_timestamps: list[dict]) -> list[list[dict]]:
    """Match subtitle chunks with timestamped words."""

    aligned_chunks = []
    word_index = 0

    for chunk in chunks:
        chunk_word_count = len(chunk.split())

        chunk_words = word_timestamps[word_index:word_index + chunk_word_count]

        aligned_chunks.append(chunk_words)

        word_index += chunk_word_count

    return aligned_chunks

### How `align_chunks_with_words()` works

- It gives each subtitle chunk the timestamped words that belong to it.

- For each chunk, it counts the words, then takes that many words from `word_timestamps`, in order.

- `word_index` remembers where the next chunk starts.

- It matches words by position, not by spelling. If Whisper splits or merges words differently from the script (for example numbers or names), later chunks can get the wrong timestamps.

- Example input:

```python
subtitle_chunks = ["AI is changing the world", "and helping people work"]

word_timestamps = [
    {"word": "AI",       "start": 0.00, "end": 0.30},
    {"word": "is",       "start": 0.30, "end": 0.42},
    {"word": "changing", "start": 0.42, "end": 0.95},
    {"word": "the",      "start": 0.95, "end": 1.10},
    {"word": "world",    "start": 1.10, "end": 1.55},
    {"word": "and",      "start": 1.55, "end": 1.75},
    {"word": "helping",  "start": 1.75, "end": 2.20},
    {"word": "people",   "start": 2.20, "end": 2.60},
    {"word": "work",     "start": 2.60, "end": 3.00}
]
```

- Output: one list of words per chunk:

```python
[
    [
        {"word": "AI",       "start": 0.00, "end": 0.30},
        {"word": "is",       "start": 0.30, "end": 0.42},
        {"word": "changing", "start": 0.42, "end": 0.95},
        {"word": "the",      "start": 0.95, "end": 1.10},
        {"word": "world",    "start": 1.10, "end": 1.55}
    ],
    [
        {"word": "and",      "start": 1.55, "end": 1.75},
        {"word": "helping",  "start": 1.75, "end": 2.20},
        {"word": "people",   "start": 2.20, "end": 2.60},
        {"word": "work",     "start": 2.60, "end": 3.00}
    ]
]
```

### 4.4 Build Subtitle Segments

In [15]:
def build_subtitle_segments(chunks: list[str], aligned_chunks: list[list[dict]]) -> list[SubtitleSegment]:
    """Create subtitle segments from aligned chunks."""

    segments = []

    for index, (chunk, chunk_words) in enumerate(zip(chunks, aligned_chunks), start = 1):
        
        if not chunk_words: # If no timestamped words were matched to this subtitle chunk, skip this chunk and continue to the next one.
            continue

        segment = SubtitleSegment(index = index,
                                    text = chunk,
                                    start_time = chunk_words[0]["start"],
                                    end_time = chunk_words[-1]["end"]
                                    )

        segments.append(segment)

    return segments

### How `build_subtitle_segments()` works

- It turns each subtitle chunk into a `SubtitleSegment` with a number, the text, and when it should appear and disappear.

- Subtitle chunks look like this:

```python
chunks = ["Artificial intelligence is changing how",
          "people work and create. AI can help",
          "people solve problems and generate new",
          "ideas."]
```

- `enumerate(..., start = 1)` numbers the segments from 1, because SRT subtitle numbers start at 1.

- `if not chunk_words: continue` skips a chunk that has no matched words, since it has no timing. Its number is skipped too, so the numbering can have a gap.

- `start_time` is when the chunk's **first** word starts: `chunk_words[0]["start"]`.

- `end_time` is when the chunk's **last** word ends: `chunk_words[-1]["end"]`.

- Example input (the output of `align_chunks_with_words()`):

```python
chunks = ["AI is changing the world", "and helping people work"]

aligned_chunks = [
    [
        {"word": "AI",       "start": 0.00, "end": 0.30},
        ...
        {"word": "world",    "start": 1.10, "end": 1.55}
    ],
    [
        {"word": "and",      "start": 1.55, "end": 1.75},
        ...
        {"word": "work",     "start": 2.60, "end": 3.00}
    ]
]
```

- Output:

```python
[
    SubtitleSegment(index = 1, start_time = 0.00, end_time = 1.55, text = "AI is changing the world"),
    SubtitleSegment(index = 2, start_time = 1.55, end_time = 3.00, text = "and helping people work")
]
```

## 5. SRT Generation

### 5.1 Format SRT Timestamps

In [16]:
def format_srt_timestamp(seconds: float) -> str:
    """Format a number of seconds as an SRT timestamp: HH:MM:SS,mmm."""

    total_milliseconds = round(seconds * 1000)
    hours, total_milliseconds = divmod(total_milliseconds, 3_600_000)
    minutes, total_milliseconds = divmod(total_milliseconds, 60_000)
    secs, milliseconds = divmod(total_milliseconds, 1_000)

    return f"{hours:02d}:{minutes:02d}:{secs:02d},{milliseconds:03d}"

### 5.2 Build SRT Content

In [17]:
# This part creates one SRT block for every subtitle segment.
# It is a standard and clean way to generate SRT files.

def build_srt_content(segments: list[SubtitleSegment]) -> str:

    blocks = [f"{segment.index}\n"                                       
                f"{format_srt_timestamp(segment.start_time)} --> "
                f"{format_srt_timestamp(segment.end_time)}\n"
                f"{segment.text}"
                for segment in segments
                ]

    return "\n\n".join(blocks) + "\n"

### 5.3 Save SRT File

In [18]:
def save_srt_file(srt_content: str, filename: str) -> Path:
    """Write SRT content to the subtitle output directory and return the file path."""

    srt_path = SUBTITLE_DIR / filename
    srt_path.write_text(srt_content, encoding = "utf-8")

    return srt_path

## 6. Generate Subtitles

### 6.1 Generate Subtitles

In [19]:
def generate_subtitles(script: VideoScript, speech_result: SpeechResult) -> SubtitleResult:
    """Generate subtitles from a video script and narration audio."""

    chunks = split_into_subtitle_chunks(script.narration)

    word_timestamps = transcribe_audio(speech_result.audio_path)

    aligned_chunks = align_chunks_with_words(chunks, word_timestamps)

    segments = build_subtitle_segments(chunks, aligned_chunks)

    srt_content = build_srt_content(segments)

    srt_path = save_srt_file(srt_content, "subtitles.srt")

    return SubtitleResult(subtitle_path = str(srt_path),
                            segments = segments,
                            duration_seconds = speech_result.duration_seconds
                            )

### 6.2 Test Subtitle Generation

In [20]:
test_script = VideoScript(title = "Agentic AI Explained",
                          narration = ("Artificial intelligence is changing how people work and create. "
                                       "AI can help people solve problems and generate new ideas."
                                       ))

In [21]:
alignment_client = create_alignment_client(provider = ALIGNMENT_PROVIDER, api_key = API_KEY)

In [22]:
test_audio_path = SUBTITLE_DIR / "test_narration.mp3"

response = alignment_client.audio.speech.create(model = "gpt-4o-mini-tts",
                                                voice = "alloy",
                                                input = test_script.narration,
                                                response_format = "mp3"
                                                )

test_audio_path.write_bytes(response.content)

147456

In [23]:
# See notebook 03, section 4.4 for the details of this function
def get_audio_duration(audio_path: Path) -> float:
    """Return the duration of an audio file in seconds."""

    result = subprocess.run(["ffprobe",
                            "-v", "error",
                            "-show_entries", "format=duration",
                            "-of", "default=noprint_wrappers=1:nokey=1",
                            str(audio_path)],
                            capture_output = True,
                            text = True,
                            check = True
                            )

    return float(result.stdout.strip())

In [24]:
speech_result = SpeechResult(audio_path = str(test_audio_path),
                            duration_seconds = get_audio_duration(test_audio_path),
                            provider = "openai",
                            voice = "alloy"
                            )

In [25]:
# PAID API CALL

subtitle_result = generate_subtitles(test_script, speech_result)

print(f"SRT file: {Path(subtitle_result.subtitle_path).resolve()}")
print(f"Segments: {len(subtitle_result.segments)}")
print(f"Duration: {subtitle_result.duration_seconds:.2f} seconds")

SRT file: /Users/yukpinglee/Desktop/agentic-video/outputs/subtitles/subtitles.srt
Segments: 3
Duration: 9.22 seconds
